In [ ]:
import importlib, subprocess, sys

def _ensure_sift():
    # Install opencv-contrib only if cv2.SIFT_create is unavailable (not needed on current Colab).
    import cv2
    if hasattr(cv2, 'SIFT_create'):
        return cv2
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-contrib-python-headless'], check=True)
    return importlib.reload(cv2)

cv2 = _ensure_sift()

import os, json, time, shutil, zipfile, warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from IPython.display import display, Markdown

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'OpenCV {cv2.__version__} | NumPy {np.__version__} | pandas {pd.__version__} | Colab: {IN_COLAB}')
assert hasattr(cv2, 'SIFT_create'), 'SIFT is not available in this OpenCV build.'

### 4.1 Experiment configuration
Every tunable quantity lives here, with its justification, so that no unexplained constants appear later in the code.

In [ ]:
# ---------------------------------------------------------------- reproducibility
SEED = 42
np.random.seed(SEED)
cv2.setRNGSeed(SEED)            # makes OpenCV's RANSAC sampling repeatable

# ---------------------------------------------------------------- data location
# 'wikimedia' : download freely licensed photos from Wikimedia Commons (default)
# 'drive'     : copy your own dataset from Google Drive (same folder layout, see Section 5)
# 'existing'  : DATA_ROOT already contains the dataset
DATA_SOURCE     = os.environ.get('LAB_DATA_SOURCE', 'wikimedia')
DATA_ROOT       = Path(os.environ.get('LAB_DATA_ROOT', '/content/landmark_lab' if IN_COLAB else 'landmark_lab'))
DRIVE_DATA_ROOT = '/content/drive/MyDrive/landmark_lab'   # used only when DATA_SOURCE == 'drive'
FORCE_DOWNLOAD  = False                                   # True = discard cached images and download again

DB_DIR      = DATA_ROOT / 'landmark_db'          # landmark_db/<Landmark_Name>/ref_XX.jpg
HOLDOUT_DIR = DATA_ROOT / 'queries' / 'holdout'  # queries/holdout/<Landmark_Name>.jpg (never in the DB)
UNKNOWN_DIR = DATA_ROOT / 'queries' / 'unknown'  # queries/unknown/*.jpg (landmarks not in the DB)
OUT_DIR     = DATA_ROOT / 'outputs'
FIG_DIR     = OUT_DIR / 'screenshots'

# Search phrases per landmark, tried in order until enough photos are found.
LANDMARK_QUERIES = {
    'Eiffel_Tower':        ['Eiffel Tower Paris', 'Tour Eiffel Champ de Mars'],
    'Taj_Mahal':           ['Taj Mahal Agra', 'Taj Mahal front view'],
    'Colosseum':           ['Colosseum Rome exterior', 'Colosseo Roma'],
    'Statue_of_Liberty':   ['Statue of Liberty New York', 'Statue of Liberty Liberty Island'],
    'Great_Wall_of_China': ['Great Wall of China Mutianyu', 'Great Wall of China Badaling'],
}
UNKNOWN_QUERIES = {'Sydney_Opera_House': ['Sydney Opera House', 'Sydney Opera House harbour']}
N_REFERENCE = 5    # reference photos per landmark in the database
N_HOLDOUT   = 1    # held-out photo per landmark (real different-viewpoint query)
N_UNKNOWN   = 2    # photos of the landmark that is NOT in the database
THUMB_WIDTH = 960  # Wikimedia serves pre-rendered thumbnails at standard widths such as 960 px

# ---------------------------------------------------------------- preprocessing / SIFT
MAX_IMAGE_DIM   = 800    # longest side after loading; bounds runtime while keeping detail
SIFT_N_FEATURES = 2000   # keep the 2000 strongest keypoints per image (0 would mean 'all')

# ---------------------------------------------------------------- matching / decision
LOWE_RATIO              = 0.75  # ratio-test threshold tau (Section 2.4)
MIN_GOOD_MATCHES        = 20    # T_m: minimum reliable matches to accept a candidate
MIN_INLIERS             = 15    # T_i: minimum RANSAC inliers to accept a candidate
MIN_HOMOGRAPHY_MATCHES  = 10    # below this RANSAC is not attempted (4 is the mathematical minimum)
RANSAC_REPROJ_THRESHOLD = 5.0   # px; inlier if reprojection error < 5 px
TOP_K_VERIFY            = 3     # candidates that receive geometric verification

# ---------------------------------------------------------------- evaluation conditions
ROTATION_DEG      = 45
RESIZE_FACTOR     = 0.5
BRIGHTNESS_FACTOR = 0.35
PERSPECTIVE_SHIFT = 0.18   # fraction of width by which the top corners move inward

# ---------------------------------------------------------------- plotting
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
ACCEPT_COLOR, REJECT_COLOR, MUTED = '#0ca30c', '#d03b3b', '#9a9993'
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'axes.edgecolor': '#52514e', 'axes.labelcolor': '#0b0b0b',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

for d in (DB_DIR, HOLDOUT_DIR, UNKNOWN_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert 0 < LOWE_RATIO < 1 and MIN_HOMOGRAPHY_MATCHES >= 4
print('Data root:', DATA_ROOT.resolve(), '| source:', DATA_SOURCE)

## 5. Data Acquisition and Preparation — *Task 1: Landmark Image Database*

The database is a folder per landmark, exactly as the assignment requests:

```
landmark_lab/
├── landmark_db/
│   ├── Eiffel_Tower/          ref_01.jpg … ref_05.jpg
│   ├── Taj_Mahal/             ref_01.jpg … ref_05.jpg
│   ├── Colosseum/             …
│   ├── Statue_of_Liberty/     …
│   └── Great_Wall_of_China/   …
├── queries/
│   ├── holdout/   Eiffel_Tower.jpg, Taj_Mahal.jpg, …   (never used as references)
│   └── unknown/   Sydney_Opera_House_01.jpg, …         (landmark not in the database)
└── outputs/       results table, descriptor database, screenshots
```

**Default — automatic download.** The cell below queries the public Wikimedia Commons API for photographs of each landmark, keeps JPEG photographs with a reasonable size and aspect ratio, skips near-duplicates, and saves them in the layout above. All images are freely licensed; their source pages are written to `outputs/image_manifest.csv` for attribution. Downloads are cached: re-running the cell does not download again.

**Alternative — your own images.** Put your photos in the same layout under `MyDrive/landmark_lab/` and set `DATA_SOURCE = 'drive'` in the configuration cell. The rest of the notebook discovers landmark names from the folder names, so any set of five or more landmarks works.

> **Manual check (important).** Automatic search can occasionally return a drawing, an interior shot or a souvenir photo. Inspect the contact sheet printed after the download; delete any unsuitable file from `landmark_db/` (Colab's file browser on the left) and re-run the download cell — it tops the folder up with the next search results.

In [ ]:
COMMONS_API = 'https://commons.wikimedia.org/w/api.php'
HTTP_HEADERS = {'User-Agent': 'SIFT-Landmark-Lab/1.0 (university coursework; Google Colab notebook)'}

def commons_search(query: str, limit: int = 40) -> list:
    '''Return candidate photos (title, thumbnail URL, source page) for a Wikimedia Commons search.'''
    params = {
        'action': 'query', 'format': 'json', 'generator': 'search',
        'gsrsearch': f'{query} filetype:bitmap', 'gsrnamespace': 6, 'gsrlimit': limit,
        'prop': 'imageinfo', 'iiprop': 'url|size|mime', 'iiurlwidth': THUMB_WIDTH,
    }
    r = requests.get(COMMONS_API, params=params, headers=HTTP_HEADERS, timeout=30)
    r.raise_for_status()
    pages = sorted(r.json().get('query', {}).get('pages', {}).values(), key=lambda p: p.get('index', 10**9))
    candidates = []
    for p in pages:
        info = (p.get('imageinfo') or [{}])[0]
        w, h = info.get('width', 0), info.get('height', 0)
        if info.get('mime') != 'image/jpeg' or min(w, h) < 500 or not (0.5 <= w / max(h, 1) <= 2.0):
            continue                                    # photographs only, no tiny or panoramic files
        candidates.append({'title': p['title'], 'url': info.get('thumburl') or info.get('url'),
                           'source_page': info.get('descriptionurl', '')})
    return candidates

def fetch_image(url: str, retries: int = 4) -> Optional[np.ndarray]:
    '''Download and decode an image; polite exponential back-off on rate limiting.'''
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HTTP_HEADERS, timeout=40)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** (attempt + 1)); continue
            r.raise_for_status()
            img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
            return img
        except requests.RequestException:
            time.sleep(2 ** (attempt + 1))
    return None

def tiny_signature(img: np.ndarray) -> np.ndarray:
    '''16x16 grayscale thumbnail used to detect near-duplicate downloads.'''
    return cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (16, 16), interpolation=cv2.INTER_AREA).astype(np.float32)

def collect_photos(name: str, queries: list, n_needed: int, existing: list) -> list:
    '''Return up to n_needed new (image, record) pairs that are not duplicates of `existing` images.'''
    seen_sigs = [tiny_signature(im) for im in existing]
    picked, seen_titles = [], set()
    for q in queries:
        try:
            candidates = commons_search(q)
        except requests.RequestException as e:
            print(f'  search failed for "{q}": {e}'); continue
        for c in candidates:
            if len(picked) >= n_needed:
                return picked
            if c['title'] in seen_titles:
                continue
            seen_titles.add(c['title'])
            img = fetch_image(c['url'])
            time.sleep(0.3)                              # be polite to the public API
            if img is None or min(img.shape[:2]) < 300:
                continue
            sig = tiny_signature(img)
            if any(np.mean(np.abs(sig - s)) < 6 for s in seen_sigs):
                continue                                 # near-duplicate of an image we already have
            seen_sigs.append(sig)
            picked.append((img, {'landmark': name, 'query': q, **c}))
    return picked

def build_dataset_from_wikimedia():
    manifest_path = OUT_DIR / 'image_manifest.csv'
    manifest = pd.read_csv(manifest_path).to_dict('records') if manifest_path.exists() else []
    for name, queries in LANDMARK_QUERIES.items():
        folder = DB_DIR / name
        folder.mkdir(parents=True, exist_ok=True)
        refs = sorted(folder.glob('*.jpg'))
        holdout = HOLDOUT_DIR / f'{name}.jpg'
        need = (N_REFERENCE - len(refs)) + (0 if holdout.exists() else N_HOLDOUT)
        if need <= 0:
            print(f'{name:22s} cached ({len(refs)} references + hold-out)'); continue
        existing = [cv2.imread(str(p)) for p in refs] + ([cv2.imread(str(holdout))] if holdout.exists() else [])
        new = collect_photos(name, queries, need, existing)
        for img, rec in new:
            refs = sorted(folder.glob('*.jpg'))
            if len(refs) < N_REFERENCE:
                idx = 1
                while (folder / f'ref_{idx:02d}.jpg').exists():
                    idx += 1
                out = folder / f'ref_{idx:02d}.jpg'; role = 'reference'
            else:
                out = holdout; role = 'holdout'
            cv2.imwrite(str(out), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
            manifest.append({**rec, 'role': role, 'file': str(out.relative_to(DATA_ROOT))})
        print(f'{name:22s} {len(sorted(folder.glob("*.jpg")))} references, hold-out: {holdout.exists()}')
    for name, queries in UNKNOWN_QUERIES.items():
        have = sorted(UNKNOWN_DIR.glob(f'{name}_*.jpg'))
        need = N_UNKNOWN - len(have)
        if need > 0:
            for img, rec in collect_photos(name, queries, need, [cv2.imread(str(p)) for p in have]):
                idx = len(sorted(UNKNOWN_DIR.glob(f'{name}_*.jpg'))) + 1
                out = UNKNOWN_DIR / f'{name}_{idx:02d}.jpg'
                cv2.imwrite(str(out), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
                manifest.append({**rec, 'role': 'unknown', 'file': str(out.relative_to(DATA_ROOT))})
        print(f'{name:22s} {len(sorted(UNKNOWN_DIR.glob(f"{name}_*.jpg")))} unknown-landmark queries')
    pd.DataFrame(manifest).to_csv(manifest_path, index=False)

if DATA_SOURCE == 'wikimedia':
    if FORCE_DOWNLOAD and DATA_ROOT.exists():
        for d in (DB_DIR, HOLDOUT_DIR, UNKNOWN_DIR):
            shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True, exist_ok=True)
    build_dataset_from_wikimedia()
elif DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copytree(DRIVE_DATA_ROOT, DATA_ROOT, dirs_exist_ok=True)
    print('Copied dataset from', DRIVE_DATA_ROOT)
elif DATA_SOURCE == 'existing':
    print('Using the dataset already present in', DATA_ROOT)
else:
    raise ValueError(f'Unknown DATA_SOURCE: {DATA_SOURCE!r}')

### 5.1 Dataset validation
The landmark labels are taken from the folder names. The checks below stop the notebook early, with a clear message, if the database does not satisfy the assignment (≥ 5 landmarks, several images each) or if a file cannot be decoded.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(folder: Path) -> list:
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT) if folder.exists() else []

LANDMARKS = sorted(d.name for d in DB_DIR.iterdir() if d.is_dir() and list_images(d))
inventory = pd.DataFrame([{
    'Landmark': lm.replace('_', ' '),
    'Reference images': len(list_images(DB_DIR / lm)),
    'Hold-out query': 'yes' if list_images(HOLDOUT_DIR) and any(p.stem == lm for p in list_images(HOLDOUT_DIR)) else 'no',
} for lm in LANDMARKS])
display(inventory.style.hide(axis='index'))
print('Unknown-landmark queries:', [p.name for p in list_images(UNKNOWN_DIR)])

assert len(LANDMARKS) >= 5, f'The assignment requires at least 5 landmarks; found {len(LANDMARKS)}. Check the download/Drive folder.'
assert all(len(list_images(DB_DIR / lm)) >= 2 for lm in LANDMARKS), 'Each landmark needs multiple reference images.'
unreadable = [p for lm in LANDMARKS for p in list_images(DB_DIR / lm) if cv2.imread(str(p)) is None]
assert not unreadable, f'Unreadable images: {unreadable}'
LANDMARK_COLORS = {lm: PALETTE[i % len(PALETTE)] for i, lm in enumerate(LANDMARKS)}

In [ ]:
def load_image(path, max_dim: int = MAX_IMAGE_DIM) -> np.ndarray:
    '''Read a colour image and downscale it so that its longest side is at most max_dim (never upscales).'''
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f'Cannot read image: {path}')
    scale = max_dim / max(img.shape[:2])
    if scale < 1.0:
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    return img

def rgb(img_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def nice(name: str) -> str:
    return name.replace('_', ' ')

def savefig(fig, name: str):
    fig.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight', facecolor='white')

# Contact sheet of the database: one row per landmark
n_cols = max(len(list_images(DB_DIR / lm)) for lm in LANDMARKS)
fig, axes = plt.subplots(len(LANDMARKS), n_cols, figsize=(2.6 * n_cols, 2.2 * len(LANDMARKS)), squeeze=False)
for r, lm in enumerate(LANDMARKS):
    paths = list_images(DB_DIR / lm)
    for c in range(n_cols):
        ax = axes[r, c]; ax.axis('off')
        if c < len(paths):
            ax.imshow(rgb(load_image(paths[c], 400)))
            ax.set_title(paths[c].name, fontsize=8, fontweight='normal')
    axes[r, 0].text(-0.08, 0.5, nice(lm), transform=axes[r, 0].transAxes, rotation=90,
                    ha='right', va='center', fontsize=10, fontweight='bold')
fig.suptitle('Task 1 — Reference database (one folder per landmark)', fontsize=13, fontweight='bold')
fig.tight_layout(); savefig(fig, 'task1_database_contact_sheet'); plt.show()

**Preprocessing choices.** Images are read in colour (for display) and downscaled so the longest side is at most 800 px. This bounds the brute-force matching cost — which grows with the *product* of the two keypoint counts — and makes the reference images comparable in resolution. Downscaling never removes the information SIFT needs at landmark scale; it only discards the finest octave, which mostly contains noise and texture that changes between photographs anyway. Grayscale conversion happens inside the feature extractor, because SIFT is defined on intensity.

## 6. SIFT Feature Extraction — *Task 2*

For every reference image: (1) load and convert to grayscale, (2) detect SIFT keypoints, (3) compute their 128-D descriptors, (4) display the keypoints, (5) store descriptors together with their landmark label.

`cv2.SIFT_create(nfeatures=2000)` keeps the 2000 keypoints with the strongest DoG response; all other parameters keep Lowe's recommended defaults (3 layers per octave, contrast threshold 0.04, edge threshold 10, initial $\sigma = 1.6$).

In [ ]:
SIFT = cv2.SIFT_create(nfeatures=SIFT_N_FEATURES)

@dataclass
class ReferenceEntry:
    landmark: str
    path: Path
    image: np.ndarray
    keypoints: tuple
    descriptors: np.ndarray

def extract_sift(img_bgr: np.ndarray):
    '''Grayscale conversion + SIFT detection/description. Returns (gray, keypoints, descriptors or None).'''
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    keypoints, descriptors = SIFT.detectAndCompute(gray, None)
    return gray, keypoints, descriptors

reference_db = []
t0 = time.perf_counter()
for lm in LANDMARKS:
    for path in list_images(DB_DIR / lm):
        img = load_image(path)
        _, kps, desc = extract_sift(img)
        if desc is None or len(kps) < MIN_HOMOGRAPHY_MATCHES:
            warnings.warn(f'{path} yields only {len(kps)} keypoints and is skipped.'); continue
        assert desc.shape == (len(kps), 128) and desc.dtype == np.float32
        reference_db.append(ReferenceEntry(lm, path, img, kps, desc))
build_time = time.perf_counter() - t0

db_summary = (pd.DataFrame([{'Landmark': nice(e.landmark), 'Keypoints': len(e.keypoints)} for e in reference_db])
              .groupby('Landmark')['Keypoints'].agg(['count', 'sum', 'mean', 'min', 'max']).round(0).astype(int)
              .rename(columns={'count': 'Images', 'sum': 'Total descriptors', 'mean': 'Mean / image',
                               'min': 'Min', 'max': 'Max'}))
display(db_summary)
print(f'{len(reference_db)} reference images encoded in {build_time:.2f} s')

### 6.1 Visualising the detected keypoints
Keypoints are drawn with `DRAW_RICH_KEYPOINTS`: the **circle radius is the keypoint's scale** and the **radial line its orientation** — the two quantities that make the descriptor scale- and rotation-invariant. For legibility, only the 250 strongest keypoints are drawn; the title gives the total.

In [ ]:
def draw_keypoints(img_bgr, keypoints, max_draw: int = 250, color=(0, 220, 0)) -> np.ndarray:
    strongest = sorted(keypoints, key=lambda k: k.response, reverse=True)[:max_draw]
    return cv2.drawKeypoints(img_bgr, strongest, None, color=color, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

per_lm = {lm: [e for e in reference_db if e.landmark == lm] for lm in LANDMARKS}
n_cols = max(len(v) for v in per_lm.values())
fig, axes = plt.subplots(len(LANDMARKS), n_cols, figsize=(2.9 * n_cols, 2.5 * len(LANDMARKS)), squeeze=False)
for r, lm in enumerate(LANDMARKS):
    for c in range(n_cols):
        ax = axes[r, c]; ax.axis('off')
        if c < len(per_lm[lm]):
            e = per_lm[lm][c]
            ax.imshow(rgb(draw_keypoints(e.image, e.keypoints)))
            ax.set_title(f'{e.path.name} — {len(e.keypoints)} kp', fontsize=8, fontweight='normal')
    axes[r, 0].text(-0.08, 0.5, nice(lm), transform=axes[r, 0].transAxes, rotation=90,
                    ha='right', va='center', fontsize=10, fontweight='bold')
fig.suptitle('Task 2 — SIFT keypoints on every reference image (scale = radius, orientation = line)',
             fontsize=13, fontweight='bold')
fig.tight_layout(); savefig(fig, 'task2_reference_keypoints'); plt.show()

### 6.2 Storing descriptors with their landmark labels
The descriptor database is held in memory (`reference_db`) and also written to disk as a compressed NumPy archive: every descriptor row carries its landmark label and the index of the image it came from, together with the keypoint geometry needed later for homography estimation. This makes the database reusable without recomputing features.

In [ ]:
desc_all   = np.vstack([e.descriptors for e in reference_db])
labels_all = np.concatenate([[e.landmark] * len(e.keypoints) for e in reference_db])
img_idx    = np.concatenate([[i] * len(e.keypoints) for i, e in enumerate(reference_db)])
kp_geom    = np.vstack([[(k.pt[0], k.pt[1], k.size, k.angle, k.response) for k in e.keypoints] for e in reference_db])
np.savez_compressed(OUT_DIR / 'sift_descriptor_database.npz', descriptors=desc_all, labels=labels_all,
                    image_index=img_idx, keypoint_xy_size_angle_response=kp_geom,
                    image_paths=np.array([str(e.path) for e in reference_db]))

assert desc_all.shape == (len(labels_all), 128)
chk = np.load(OUT_DIR / 'sift_descriptor_database.npz')
assert np.array_equal(chk['descriptors'], desc_all) and set(chk['labels']) == set(LANDMARKS)
print(f'Stored {desc_all.shape[0]:,} descriptors x {desc_all.shape[1]} dims '
      f'({desc_all.nbytes / 1e6:.1f} MB) for {len(set(labels_all))} landmarks -> outputs/sift_descriptor_database.npz')

## 7. Query Image Processing — *Task 3*

A query is processed with exactly the same pipeline as the references — load, grayscale, detect, describe — so that descriptors are comparable. The function accepts a file path, an uploaded file, or an image array (the synthetic test conditions in Section 11 are arrays).

In [ ]:
@dataclass
class QueryFeatures:
    image: np.ndarray
    gray: np.ndarray
    keypoints: tuple
    descriptors: Optional[np.ndarray]

def process_query(query) -> QueryFeatures:
    '''Task 3: load (path or BGR array) -> grayscale -> SIFT keypoints -> descriptors.'''
    img = load_image(query) if isinstance(query, (str, Path)) else query
    if img is None or img.ndim != 3:
        raise ValueError('Query must be a path or a BGR colour image array.')
    gray, kps, desc = extract_sift(img)
    return QueryFeatures(img, gray, kps, desc)

def show_query(qf: QueryFeatures, title: str = 'Query', save_as: Optional[str] = None):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
    axes[0].imshow(rgb(qf.image)); axes[0].set_title('Query image')
    axes[1].imshow(qf.gray, cmap='gray'); axes[1].set_title('Grayscale (input to SIFT)')
    axes[2].imshow(rgb(draw_keypoints(qf.image, qf.keypoints))); axes[2].set_title(f'{len(qf.keypoints)} SIFT keypoints')
    for a in axes: a.axis('off')
    fig.suptitle(f'Task 3 — {title}', fontsize=13, fontweight='bold'); fig.tight_layout()
    if save_as: savefig(fig, save_as)
    plt.show()

DEMO_QUERY = next(iter(list_images(HOLDOUT_DIR)), None) or list_images(DB_DIR / LANDMARKS[0])[0]
demo_qf = process_query(DEMO_QUERY)
show_query(demo_qf, f'query "{Path(DEMO_QUERY).name}"', save_as='task3_query_keypoints')

**Optional — test your own photo.** Run the cell below in Colab, upload any landmark photograph, and the full pipeline of Sections 8–10 will be applied to it (run those sections first).

In [ ]:
RUN_UPLOAD = False   # set to True to upload your own query photo
if RUN_UPLOAD and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for fname, data in uploaded.items():
        user_img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        cv2.imwrite(str(DATA_ROOT / fname), user_img)
        res = recognize(load_image(DATA_ROOT / fname))
        show_result(res, title=f'uploaded "{fname}"', save_as=f'upload_{Path(fname).stem}')

## 8. Feature Matching and Landmark Recognition — *Task 4*

### 8.1 Brute-force $k$-NN matching with Lowe's ratio test
`cv2.BFMatcher(cv2.NORM_L2)` computes exact Euclidean distances (Section 2.4). `crossCheck` must be `False` because the ratio test requires the two nearest neighbours from `knnMatch(k=2)`.

Matching is done **query vs. each reference image separately**, not against one pooled descriptor set. Pooling would make the second-nearest neighbour often come from *another photo of the same landmark*, so a perfectly good match would fail the ratio test simply because the landmark is well represented in the database. Per-image matching also gives us a single, geometrically consistent reference image for the homography in Task 5.

### 8.2 Ranking landmarks
A landmark's score is the number of reliable matches with its **best** reference image (the maximum over its references). The maximum, rather than the sum, is used because a query photograph normally corresponds to one viewpoint; summing would reward landmarks that simply have more reference photos.

### 8.3 Decision rule (open-set rejection)
The top-ranked candidates (`TOP_K_VERIFY = 3`) are geometrically verified (Section 9), and the candidate with the most RANSAC inliers is selected. It is **accepted only if**

$$ \text{reliable matches} \ge T_m = 20 \quad\text{and}\quad \text{RANSAC inliers} \ge T_i = 15 . $$

*Why these values?* Even an unrelated photograph produces a few matches that pass the ratio test by coincidence, and with 2000 query descriptors compared against 25 reference images, the maximum of these chance counts is typically in the single digits to low teens. Chance matches also rarely agree on one homography, so RANSAC can usually find only a handful of mutually consistent ones (any 4 points fit a homography exactly, so small inlier counts are meaningless). Requiring 20 matches and 15 inliers puts the acceptance boundary clearly above that chance level while remaining reachable for a genuine view of the landmark. Requiring both conditions — and not merely being the top-ranked landmark — is what allows an unknown landmark to be **rejected**.

In [ ]:
BF_MATCHER = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

def ratio_test_matches(desc_query, desc_ref, ratio: float = LOWE_RATIO) -> list:
    '''k-NN (k=2) Euclidean matching followed by Lowe's ratio test.
    Returned DMatch objects: queryIdx -> query keypoint, trainIdx -> reference keypoint.'''
    if desc_query is None or desc_ref is None or len(desc_query) < 2 or len(desc_ref) < 2:
        return []
    good = []
    for pair in BF_MATCHER.knnMatch(desc_query, desc_ref, k=2):
        if len(pair) < 2:
            continue
        best, second = pair
        if best.distance < ratio * second.distance:
            good.append(best)
    return good

@dataclass
class Candidate:
    landmark: str
    good_matches: int                  # reliable matches with the best reference image
    best_ref: ReferenceEntry
    matches: list                      # DMatch list for best_ref
    per_reference: list                # reliable matches with every reference image of this landmark
    inliers: int = 0
    geometry: Optional[dict] = None

def rank_landmarks(qf: QueryFeatures) -> list:
    '''Task 4.1-4.5: match against every reference image and rank landmarks by reliable matches.'''
    by_landmark = {}
    for entry in reference_db:
        m = ratio_test_matches(qf.descriptors, entry.descriptors)
        by_landmark.setdefault(entry.landmark, []).append((len(m), entry, m))
    ranking = []
    for lm, lst in by_landmark.items():
        n, entry, m = max(lst, key=lambda t: t[0])
        ranking.append(Candidate(lm, n, entry, m, [t[0] for t in lst]))
    return sorted(ranking, key=lambda c: c.good_matches, reverse=True)

ranking_demo = rank_landmarks(demo_qf)
display(pd.DataFrame([{'Rank': i + 1, 'Landmark': nice(c.landmark), 'Reliable matches (best reference)': c.good_matches,
                       'Best reference': c.best_ref.path.name, 'Matches per reference': c.per_reference}
                      for i, c in enumerate(ranking_demo)]).style.hide(axis='index'))

## 9. Landmark Localization — *Task 5*

For a candidate landmark, the reliable correspondences between its best reference image and the query are used to estimate a homography with RANSAC. The four corners of the reference image are projected into the query with $\mathbf{H}$ and joined into a polygon.

**When is the transformation considered unreliable?** Besides having too few matches or inliers, a homography can be numerically valid yet physically meaningless. The polygon is drawn only if all of the following hold, otherwise a message is shown instead:

| Check | Reason |
|---|---|
| at least `MIN_HOMOGRAPHY_MATCHES` (10) reliable matches and `MIN_INLIERS` (15) inliers | too little evidence for a stable 8-DOF estimate |
| $\det(\mathbf{H}_{2\times2}) > 0$ and within $[0.01, 100]$ | negative = mirror image; extreme = collapse or explosion of the region |
| projected quadrilateral is convex | a real view of a planar region cannot fold over itself |
| projected area ≥ 1 % of the query and corners stay within a generous margin around the query | rejects degenerate, near-singular solutions |

Note what the polygon means: it outlines **the region of the query that corresponds to the whole reference photograph**, under a planar approximation. When the reference photo is tightly framed on the landmark, this is the landmark; when it includes surroundings, the polygon includes them too.

In [ ]:
def homography_is_plausible(H: np.ndarray, polygon: np.ndarray, query_shape) -> tuple:
    qh, qw = query_shape[:2]
    det = float(np.linalg.det(H[:2, :2]))
    if not (0.01 < det < 100):
        return False, f'degenerate transformation (det of linear part = {det:.3g})'
    poly = polygon.reshape(-1, 2).astype(np.float32)
    if not cv2.isContourConvex(poly):
        return False, 'projected region is not convex (folded / twisted)'
    area_frac = cv2.contourArea(poly) / float(qw * qh)
    if area_frac < 0.01:
        return False, f'projected region too small ({area_frac:.2%} of the image)'
    margin = 1.0
    if (poly[:, 0].min() < -margin * qw or poly[:, 0].max() > (1 + margin) * qw or
            poly[:, 1].min() < -margin * qh or poly[:, 1].max() > (1 + margin) * qh):
        return False, 'projected corners lie far outside the query image'
    return True, 'ok'

def localize(qf: QueryFeatures, cand: Candidate) -> dict:
    '''Task 5: RANSAC homography from reference -> query and projection of the reference corners.'''
    result = {'H': None, 'mask': None, 'inliers': 0, 'polygon': None, 'valid': False, 'reason': ''}
    if len(cand.matches) < MIN_HOMOGRAPHY_MATCHES:
        result['reason'] = f'only {len(cand.matches)} reliable matches (< {MIN_HOMOGRAPHY_MATCHES}); homography not attempted'
        return result
    src = np.float32([cand.best_ref.keypoints[m.trainIdx].pt for m in cand.matches]).reshape(-1, 1, 2)
    dst = np.float32([qf.keypoints[m.queryIdx].pt for m in cand.matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, RANSAC_REPROJ_THRESHOLD)
    if H is None or mask is None:
        result['reason'] = 'RANSAC could not find a homography'
        return result
    result.update(H=H, mask=mask.ravel().astype(bool), inliers=int(mask.sum()))
    h, w = cand.best_ref.image.shape[:2]
    corners = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]]).reshape(-1, 1, 2)
    polygon = cv2.perspectiveTransform(corners, H)
    result['polygon'] = polygon
    if result['inliers'] < MIN_INLIERS:
        result['reason'] = f"only {result['inliers']} RANSAC inliers (< {MIN_INLIERS})"
        return result
    result['valid'], result['reason'] = homography_is_plausible(H, polygon, qf.image.shape)
    return result

### 9.1 The complete recogniser
`recognize()` chains Tasks 3–5 and applies the decision rule from Section 8.3. The processing time covers feature extraction, matching against the whole database, and geometric verification — everything a deployed system would compute — and excludes plotting.

In [ ]:
@dataclass
class RecognitionResult:
    query: QueryFeatures
    ranking: list
    prediction: Optional[Candidate]       # best verified candidate (may still be rejected)
    recognized: bool
    reason: str
    seconds: float

    @property
    def label(self) -> str:
        return self.prediction.landmark if (self.recognized and self.prediction) else 'REJECTED'

def recognize(query) -> RecognitionResult:
    t0 = time.perf_counter()
    qf = process_query(query)
    if qf.descriptors is None or len(qf.keypoints) < MIN_HOMOGRAPHY_MATCHES:
        return RecognitionResult(qf, [], None, False, 'too few keypoints in the query', time.perf_counter() - t0)
    ranking = rank_landmarks(qf)
    for cand in ranking[:TOP_K_VERIFY]:
        cand.geometry = localize(qf, cand)
        cand.inliers = cand.geometry['inliers']
    best = max(ranking[:TOP_K_VERIFY], key=lambda c: (c.inliers, c.good_matches))
    ok_matches = best.good_matches >= MIN_GOOD_MATCHES
    ok_inliers = best.inliers >= MIN_INLIERS
    recognized = ok_matches and ok_inliers
    if recognized:
        reason = f'{best.good_matches} reliable matches >= {MIN_GOOD_MATCHES} and {best.inliers} inliers >= {MIN_INLIERS}'
    else:
        failed = []
        if not ok_matches: failed.append(f'{best.good_matches} reliable matches < {MIN_GOOD_MATCHES}')
        if not ok_inliers: failed.append(f'{best.inliers} inliers < {MIN_INLIERS}')
        reason = 'insufficient evidence: ' + '; '.join(failed)
    return RecognitionResult(qf, ranking, best, recognized, reason, time.perf_counter() - t0)

### 9.2 Result display
One figure answers every item of the assignment's *Expected Output* list: (1) the query image, (2) its SIFT keypoints, (3) the matched features between query and reference — RANSAC inliers in green, rejected outliers in red, (4) the predicted landmark, (5) the number of reliable matches (ranking chart with the acceptance threshold), (6) the localised region when verification succeeds, and (7) the recognised/rejected message.

In [ ]:
def draw_match_panel(qf: QueryFeatures, cand: Candidate, max_lines: int = 60) -> np.ndarray:
    '''Side-by-side query | reference; inliers green, outliers red (subsampled for legibility).'''
    mask = cand.geometry['mask'] if cand.geometry and cand.geometry['mask'] is not None else np.zeros(len(cand.matches), bool)
    rng = np.random.default_rng(SEED)
    inl = [m for m, k in zip(cand.matches, mask) if k]
    out = [m for m, k in zip(cand.matches, mask) if not k]
    inl = list(rng.choice(inl, min(len(inl), max_lines), replace=False)) if inl else []
    out = list(rng.choice(out, min(len(out), max_lines // 3), replace=False)) if out else []
    canvas = cv2.drawMatches(qf.image, qf.keypoints, cand.best_ref.image, cand.best_ref.keypoints, out, None,
                             matchColor=(60, 60, 220), singlePointColor=None,
                             flags=cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS)
    canvas = cv2.drawMatches(qf.image, qf.keypoints, cand.best_ref.image, cand.best_ref.keypoints, inl, canvas,
                             matchColor=(40, 200, 40), singlePointColor=None,
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_OVER_OUTIMG | cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS)
    return canvas

def draw_localization(qf: QueryFeatures, cand: Optional[Candidate]) -> tuple:
    img = qf.image.copy()
    geo = cand.geometry if cand else None
    if geo and geo['valid']:
        thick = max(2, int(round(max(img.shape[:2]) / 200)))
        cv2.polylines(img, [np.int32(geo['polygon'])], True, (0, 0, 230), thick, cv2.LINE_AA)
        return img, 'Landmark region localized'
    return img, 'Localization unreliable — polygon not drawn' + (f"\n({geo['reason']})" if geo else '')

def show_result(res: RecognitionResult, title: str = '', save_as: Optional[str] = None,
                ground_truth: Optional[str] = None, show: bool = True):
    qf, cand = res.query, res.prediction
    fig = plt.figure(figsize=(16, 9.2))
    gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1.05], hspace=0.28, wspace=0.12)
    ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(rgb(qf.image)); ax0.set_title('(1) Query image'); ax0.axis('off')
    ax1 = fig.add_subplot(gs[0, 1]); ax1.imshow(rgb(draw_keypoints(qf.image, qf.keypoints)))
    ax1.set_title(f'(2) SIFT keypoints: {len(qf.keypoints)}'); ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 2])
    if res.ranking:
        names = [nice(c.landmark) for c in res.ranking][::-1]
        vals = [c.good_matches for c in res.ranking][::-1]
        cols = [LANDMARK_COLORS[c.landmark] if (cand and c.landmark == cand.landmark) else '#c9c8c2' for c in res.ranking][::-1]
        bars = ax2.barh(names, vals, color=cols, height=0.62, edgecolor='white', linewidth=2)
        ax2.axvline(MIN_GOOD_MATCHES, color='#52514e', ls='--', lw=1)
        ax2.text(MIN_GOOD_MATCHES, 1.01, f'T_m = {MIN_GOOD_MATCHES}', transform=ax2.get_xaxis_transform(),
                 fontsize=8, color='#52514e', ha='center', va='bottom')
        for b, v in zip(bars, vals):
            ax2.text(b.get_width(), b.get_y() + b.get_height() / 2, f' {v}', va='center', fontsize=9)
        ax2.set_xlabel('Reliable matches (best reference image)')
        ax2.set_xlim(0, max(max(vals), MIN_GOOD_MATCHES) * 1.18)
    ax2.set_title('(4–5) Landmark ranking', pad=16)

    ax3 = fig.add_subplot(gs[1, 0:2])
    if cand is not None:
        ax3.imshow(rgb(draw_match_panel(qf, cand)))
        ax3.set_title(f'(3) Matches: query (left) vs best reference "{cand.best_ref.path.name}" of {nice(cand.landmark)} '
                      f'— green = RANSAC inliers, red = outliers (subsampled)', fontsize=10)
    ax3.axis('off')

    ax4 = fig.add_subplot(gs[1, 2])
    loc_img, loc_msg = draw_localization(qf, cand)
    ax4.imshow(rgb(loc_img)); ax4.set_title(f'(6) {loc_msg}', fontsize=9.5); ax4.axis('off')

    status = 'RECOGNIZED' if res.recognized else 'REJECTED'
    colr = ACCEPT_COLOR if res.recognized else REJECT_COLOR
    pred = nice(cand.landmark) if (res.recognized and cand) else 'none (unknown / insufficient evidence)'
    lines = [f'Predicted landmark : {pred}',
             f'Reliable matches   : {cand.good_matches if cand else 0}',
             f'RANSAC inliers     : {cand.inliers if cand else 0}',
             f'Processing time    : {res.seconds:.3f} s']
    if ground_truth: lines.insert(0, f'Ground truth       : {nice(ground_truth)}')
    fig.text(0.5, 0.965, f'(7) {status}' + (f' — {title}' if title else ''), ha='center', fontsize=14,
             fontweight='bold', color=colr)
    fig.text(0.5, 0.925, '   |   '.join(lines) + f'\nDecision: {res.reason}', ha='center', va='top',
             fontsize=9.5, family='monospace', color='#0b0b0b')
    fig.subplots_adjust(top=0.86)
    if save_as: savefig(fig, save_as)
    if show:
        plt.show()
    else:
        plt.close(fig)

## 10. End-to-End Demonstration

The full pipeline applied to the demonstration query from Section 7 (a held-out photograph that is **not** in the database).

In [ ]:
demo_truth = Path(DEMO_QUERY).stem if Path(DEMO_QUERY).parent == HOLDOUT_DIR else Path(DEMO_QUERY).parent.name
demo_res = recognize(DEMO_QUERY)
show_result(demo_res, title=f'demo query "{Path(DEMO_QUERY).name}"', save_as='task4_5_demo_result', ground_truth=demo_truth)

display(pd.DataFrame([{'Rank': i + 1, 'Landmark': nice(c.landmark), 'Reliable matches': c.good_matches,
                       'RANSAC inliers': c.inliers if i < TOP_K_VERIFY else '— (not verified)',
                       'Geometry': (c.geometry['reason'] if c.geometry else '—')}
                      for i, c in enumerate(demo_res.ranking)]).style.hide(axis='index'))

## 11. Performance Evaluation — *Task 6*

The test suite is generated programmatically (Section 3 explains the design). The synthetic transformations are implemented so that they change only the intended factor: rotation enlarges the canvas so that nothing is cropped, resizing uses area interpolation, the brightness change is a pure intensity scaling, and the perspective warp moves the two top corners inward to simulate looking up at the landmark from a closer position.

In [ ]:
def rotate_image(img, angle_deg):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle_deg, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2 - w / 2; M[1, 2] += nh / 2 - h / 2
    return cv2.warpAffine(img, M, (nw, nh), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0))

def resize_image(img, factor):
    return cv2.resize(img, None, fx=factor, fy=factor, interpolation=cv2.INTER_AREA)

def darken_image(img, factor):
    return cv2.convertScaleAbs(img, alpha=factor, beta=0)

def perspective_image(img, shift):
    h, w = img.shape[:2]
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([[shift * w, 0.06 * h], [(1 - shift) * w, 0.06 * h], [w, h], [0, h]])
    return cv2.warpPerspective(img, cv2.getPerspectiveTransform(src, dst), (w, h), borderValue=(0, 0, 0))

CONDITIONS = ['Original', 'Rotated', 'Resized', 'Low brightness', 'Viewpoint (synthetic)',
              'Viewpoint (real photo)', 'Unknown landmark']

def build_test_suite() -> list:
    tests = []
    for lm in LANDMARKS:
        base_path = list_images(DB_DIR / lm)[0]
        base = load_image(base_path)
        tests += [
            ('Original', lm, base, base_path.name),
            ('Rotated', lm, rotate_image(base, ROTATION_DEG), f'{base_path.name} rotated {ROTATION_DEG} deg'),
            ('Resized', lm, resize_image(base, RESIZE_FACTOR), f'{base_path.name} x{RESIZE_FACTOR}'),
            ('Low brightness', lm, darken_image(base, BRIGHTNESS_FACTOR), f'{base_path.name} intensity x{BRIGHTNESS_FACTOR}'),
            ('Viewpoint (synthetic)', lm, perspective_image(base, PERSPECTIVE_SHIFT), f'{base_path.name} perspective warp'),
        ]
        hold = [p for p in list_images(HOLDOUT_DIR) if p.stem == lm]
        if hold:
            tests.append(('Viewpoint (real photo)', lm, load_image(hold[0]), f'holdout/{hold[0].name}'))
    for p in list_images(UNKNOWN_DIR):
        tests.append(('Unknown landmark', 'Unknown', load_image(p), f'unknown/{p.name}'))
    return tests

test_suite = build_test_suite()
print(f'{len(test_suite)} test queries:', pd.Series([t[0] for t in test_suite]).value_counts().reindex(CONDITIONS).to_dict())

# Preview of the six conditions for the first landmark
preview = [t for t in test_suite if t[1] == LANDMARKS[0]] + [t for t in test_suite if t[0] == 'Unknown landmark'][:1]
fig, axes = plt.subplots(1, len(preview), figsize=(2.9 * len(preview), 3))
for ax, (cond, gt, img, desc) in zip(np.atleast_1d(axes), preview):
    ax.imshow(rgb(img)); ax.set_title(cond, fontsize=9); ax.axis('off')
fig.suptitle(f'Task 6 — test conditions (shown for {nice(LANDMARKS[0])})', fontsize=12, fontweight='bold')
fig.tight_layout(); savefig(fig, 'task6_test_conditions'); plt.show()

### 11.1 Running the experiment
Every test is passed through the identical `recognize()` function. A composite result figure is saved for **every** test to `outputs/screenshots/` (these are the screenshots required as a deliverable); to keep the notebook readable, figures are displayed inline only for the landmark chosen in `SHOW_INLINE_FOR` and for the unknown-landmark queries.

In [ ]:
SHOW_INLINE_FOR = LANDMARKS[0]
records, results = [], []
for i, (cond, gt, img, desc) in enumerate(test_suite, start=1):
    res = recognize(img)
    results.append(res)
    known = gt != 'Unknown'
    correct = (res.recognized and res.label == gt) if known else (not res.recognized)
    cand = res.prediction
    records.append({
        'Test': f'T{i:02d}', 'Condition': cond, 'Ground truth': nice(gt), 'Query': desc,
        'Top-ranked landmark': nice(res.ranking[0].landmark) if res.ranking else '—',
        'Predicted landmark': nice(res.label) if res.recognized else 'REJECTED',
        'Keypoints': len(res.query.keypoints),
        'Reliable matches': cand.good_matches if cand else 0,
        'RANSAC inliers': cand.inliers if cand else 0,
        'Localization': 'polygon' if (cand and cand.geometry and cand.geometry['valid']) else 'unreliable',
        'Decision': 'Recognized' if res.recognized else 'Rejected',
        'Correct': correct,
        'Time (s)': round(res.seconds, 3),
    })
    fname = f'task6_T{i:02d}_{gt}_{cond}'.replace(' ', '_').replace('(', '').replace(')', '')
    show_result(res, title=f'T{i:02d} {cond} — {nice(gt)}', save_as=fname, ground_truth=gt,
                show=(gt == SHOW_INLINE_FOR or not known))

results_df = pd.DataFrame(records)
results_df.to_csv(OUT_DIR / 'experimental_results.csv', index=False)
print('Saved', len(results_df), 'rows -> outputs/experimental_results.csv;',
      len(list(FIG_DIR.glob('task6_*.png'))), 'screenshots -> outputs/screenshots/')

### 11.2 Table of experimental results

In [ ]:
def _style_row(row):
    c = '#e8f5e8' if row['Correct'] else '#fbe9e9'
    return [f'background-color: {c}' if col in ('Decision', 'Correct') else '' for col in row.index]

display(results_df.drop(columns=['Query']).style.apply(_style_row, axis=1).hide(axis='index')
        .format({'Time (s)': '{:.3f}'}).set_caption('Per-test results (green = correct decision, red = error)'))

## 12. Visualization and Analysis

### 12.1 Summary by condition and overall metrics
For known landmarks a test is correct only if the system *recognises* it *and* names the right landmark. For the unknown landmark a test is correct if the system *rejects* it. Two error types are therefore distinguished: a **false rejection** (a known landmark is not recognised) and a **false acceptance** (a wrong landmark is reported — either a confusion between known landmarks or an unknown landmark accepted).

In [ ]:
summary = (results_df.groupby('Condition', sort=False)
           .agg(Tests=('Test', 'count'), Correct=('Correct', 'sum'),
                **{'Mean keypoints': ('Keypoints', 'mean'), 'Mean reliable matches': ('Reliable matches', 'mean'),
                   'Mean inliers': ('RANSAC inliers', 'mean'), 'Mean time (s)': ('Time (s)', 'mean')})
           .reindex([c for c in CONDITIONS if c in set(results_df['Condition'])]))
summary.insert(2, 'Accuracy', summary['Correct'] / summary['Tests'])
display(summary.style.format({'Accuracy': '{:.0%}', 'Mean keypoints': '{:.0f}', 'Mean reliable matches': '{:.1f}',
                              'Mean inliers': '{:.1f}', 'Mean time (s)': '{:.3f}'})
        .set_caption('Results aggregated per test condition'))

known = results_df[results_df['Ground truth'] != 'Unknown']
unknown = results_df[results_df['Ground truth'] == 'Unknown']
false_reject = int((known['Decision'] == 'Rejected').sum())
confusions = int(((known['Decision'] == 'Recognized') & ~known['Correct']).sum())
false_accept_unknown = int((unknown['Decision'] == 'Recognized').sum())
metrics = pd.DataFrame({'Value': [
    f"{known['Correct'].mean():.1%}  ({known['Correct'].sum()}/{len(known)})",
    f"{false_reject}/{len(known)}",
    f"{confusions}/{len(known)}",
    f"{(unknown['Decision'] == 'Rejected').mean():.0%}  ({(unknown['Decision'] == 'Rejected').sum()}/{len(unknown)})" if len(unknown) else 'n/a',
    f"{results_df['Correct'].mean():.1%}  ({results_df['Correct'].sum()}/{len(results_df)})",
    f"{results_df['Time (s)'].median():.3f} s (max {results_df['Time (s)'].max():.3f} s)",
]}, index=['Recognition accuracy on known landmarks', 'False rejections (known landmark rejected)',
           'Confusions (wrong known landmark accepted)', 'Rejection rate on unknown landmark',
           'Overall correct decisions', 'Median processing time per query'])
display(metrics)

### 12.2 Reliable matches and inliers per condition
Each dot is one test query, coloured by its ground-truth landmark; the dashed lines are the acceptance thresholds. A query is accepted only when it is above **both** lines, which makes visible how far each condition sits from the decision boundary.

In [ ]:
cond_order = list(summary.index)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)
rng = np.random.default_rng(SEED)
for ax, metric, thr in [(axes[0], 'Reliable matches', MIN_GOOD_MATCHES), (axes[1], 'RANSAC inliers', MIN_INLIERS)]:
    for x, cond in enumerate(cond_order):
        sub = results_df[results_df['Condition'] == cond]
        for _, row in sub.iterrows():
            gt = row['Ground truth'].replace(' ', '_')
            col = LANDMARK_COLORS.get(gt, '#3d3d3a')
            ax.scatter(x + rng.uniform(-0.18, 0.18), max(row[metric], 0.8), s=58, color=col,
                       edgecolor='white', linewidth=1.5, marker='o' if row['Correct'] else 'X', zorder=3)
        ax.hlines(sub[metric].median(), x - 0.3, x + 0.3, color='#0b0b0b', lw=2, zorder=2)
    ax.axhline(thr, color='#52514e', ls='--', lw=1)
    ax.text(-0.4, thr, f'threshold = {thr}', ha='left', va='bottom', fontsize=8, color='#52514e')
    ax.set_yscale('log'); ax.set_ylabel(f'{metric} (log scale)')
    ax.set_xticks(range(len(cond_order))); ax.set_xticklabels(cond_order, rotation=25, ha='right')
    ax.set_title(f'{metric} per test (bar = median)')
    ax.grid(axis='y', color='#ecebe7', lw=0.8, zorder=0)
handles = [plt.Line2D([], [], marker='o', ls='', color=LANDMARK_COLORS[lm], label=nice(lm), markersize=8) for lm in LANDMARKS]
handles += [plt.Line2D([], [], marker='o', ls='', color='#3d3d3a', label='Unknown (' + ', '.join(nice(k) for k in UNKNOWN_QUERIES) + ')', markersize=8),
            plt.Line2D([], [], marker='X', ls='', color='#9a9993', label='incorrect decision', markersize=8)]
fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig.suptitle('Evidence strength under each test condition', fontsize=13, fontweight='bold')
fig.tight_layout(); savefig(fig, 'analysis_matches_inliers_by_condition'); plt.show()

### 12.3 Keypoint repeatability and processing time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
k = summary['Mean keypoints']
axes[0].bar(range(len(k)), k.values, color=PALETTE[0], width=0.62, edgecolor='white', linewidth=2)
for x, v in enumerate(k.values):
    axes[0].text(x, v, f'{v:.0f}', ha='center', va='bottom', fontsize=8)
axes[0].axhline(SIFT_N_FEATURES, color='#52514e', ls='--', lw=1)
axes[0].text(len(k) - 0.5, SIFT_N_FEATURES, f'nfeatures cap = {SIFT_N_FEATURES}', ha='right', va='bottom', fontsize=8, color='#52514e')
axes[0].set_title('Mean query keypoints detected'); axes[0].set_ylabel('Keypoints')

t = [results_df.loc[results_df['Condition'] == c, 'Time (s)'].values for c in cond_order]
bp = axes[1].boxplot(t, widths=0.5, patch_artist=True, medianprops=dict(color='#0b0b0b', lw=2))
for b in bp['boxes']:
    b.set_facecolor('#cde2fb'); b.set_edgecolor(PALETTE[0])
axes[1].set_title(f'Processing time per query (database of {len(reference_db)} images)'); axes[1].set_ylabel('Seconds')
axes[0].set_xticks(range(len(cond_order)))
axes[1].set_xticks(range(1, len(cond_order) + 1))   # boxplot positions start at 1
for ax in axes:
    ax.set_xticklabels(cond_order, rotation=25, ha='right'); ax.grid(axis='y', color='#ecebe7', lw=0.8); ax.set_axisbelow(True)
fig.tight_layout(); savefig(fig, 'analysis_keypoints_and_time'); plt.show()

### 12.4 Confusion matrix (including the reject option)

In [ ]:
row_labels = [nice(l) for l in LANDMARKS] + (['Unknown'] if len(unknown) else [])
col_labels = [nice(l) for l in LANDMARKS] + ['REJECTED']
cm = pd.crosstab(pd.Categorical(results_df['Ground truth'], categories=row_labels),
                 pd.Categorical(results_df['Predicted landmark'], categories=col_labels), dropna=False)
fig, ax = plt.subplots(figsize=(1.3 * len(col_labels) + 2, 0.75 * len(row_labels) + 1.6))
im = ax.imshow(cm.values, cmap='Blues', vmin=0)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        v = cm.values[i, j]
        ax.text(j, i, str(v), ha='center', va='center', fontsize=10,
                color='white' if v > cm.values.max() * 0.55 else '#0b0b0b')
ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, rotation=30, ha='right')
ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels)
ax.set_xlabel('System output'); ax.set_ylabel('Ground truth'); ax.set_title('Confusion matrix over all test queries')
ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, fraction=0.03, label='Number of tests')
fig.tight_layout(); savefig(fig, 'analysis_confusion_matrix'); plt.show()

### 12.5 Threshold sensitivity analysis
The thresholds were fixed in advance. Here the recorded evidence of every test is re-evaluated under a range of inlier thresholds $T_i$ (with $T_m$ fixed) to show the trade-off they control: a lower threshold recognises more genuine views but risks accepting the unknown landmark; a higher one does the opposite. This analysis is *descriptive*. Choosing the threshold from this plot would mean tuning on the test set, and a fair estimate would then require a separate validation set.

In [ ]:
thresholds = np.arange(4, 61, 1)
# Did geometric verification select the TRUE landmark? (independent of the thresholds)
true_selected = np.array([results[i].prediction is not None and nice(results[i].prediction.landmark) == gt
                          for i, gt in zip(known.index, known['Ground truth'])])
recall, unk_reject = [], []
for th in thresholds:
    passes_known = ((known['Reliable matches'] >= MIN_GOOD_MATCHES) & (known['RANSAC inliers'] >= th)).values
    recall.append(np.mean(passes_known & true_selected))
    if len(unknown):
        passes_unk = (unknown['Reliable matches'] >= MIN_GOOD_MATCHES) & (unknown['RANSAC inliers'] >= th)
        unk_reject.append(np.mean(~passes_unk.values))
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(thresholds, recall, color=PALETTE[0], lw=2, label='Known landmarks correctly recognised')
if unk_reject:
    ax.plot(thresholds, unk_reject, color=PALETTE[1], lw=2, label='Unknown landmark correctly rejected')
ax.axvline(MIN_INLIERS, color='#52514e', ls='--', lw=1)
ax.text(MIN_INLIERS, 0.03, f'  chosen T_i = {MIN_INLIERS}', fontsize=8, color='#52514e')
ax.set_xlabel('Inlier threshold T_i'); ax.set_ylabel('Rate'); ax.set_ylim(-0.03, 1.05)
ax.set_title(f'Sensitivity of the decision to the inlier threshold (T_m = {MIN_GOOD_MATCHES} fixed)')
ax.grid(color='#ecebe7', lw=0.8); ax.legend(frameon=False, loc='center right')
fig.tight_layout(); savefig(fig, 'analysis_threshold_sensitivity'); plt.show()

## 13. Results and Interpretation

The paragraph below is **generated from the measured results** of this run, so it always matches the table above; the discussion that follows separates what the data shows from what theory predicts.

In [ ]:
def describe_results():
    s = summary
    lines = []
    lines.append(f"- Over **{len(results_df)} test queries**, the system made **{results_df['Correct'].sum()} correct decisions** "
                 f"({results_df['Correct'].mean():.0%}). On known landmarks it was correct in "
                 f"**{known['Correct'].sum()}/{len(known)}** cases; the unknown landmark was rejected in "
                 f"**{(unknown['Decision'] == 'Rejected').sum()}/{len(unknown)}** cases.")
    for cond in s.index:
        lines.append(f"- *{cond}*: {s.loc[cond, 'Correct']}/{s.loc[cond, 'Tests']} correct, "
                     f"mean {s.loc[cond, 'Mean reliable matches']:.0f} reliable matches and "
                     f"{s.loc[cond, 'Mean inliers']:.0f} inliers, {s.loc[cond, 'Mean keypoints']:.0f} keypoints.")
    weakest = s.drop(index=[c for c in ['Original', 'Unknown landmark'] if c in s.index])['Mean inliers'].idxmin()
    lines.append(f"- The hardest *known-landmark* condition, measured by mean inliers, was **{weakest}**.")
    errs = results_df[~results_df['Correct']]
    if len(errs):
        lines.append('- Incorrect decisions: ' + '; '.join(
            f"{r['Test']} ({r['Condition']}: {r['Ground truth']} → {r['Predicted landmark']}, "
            f"{r['Reliable matches']} matches / {r['RANSAC inliers']} inliers)"
            for _, r in errs.iterrows()) + '.')
    else:
        lines.append('- No incorrect decisions were made in this run.')
    lines.append(f"- Median processing time was **{results_df['Time (s)'].median():.2f} s** per query against "
                 f"{len(reference_db)} reference images on this machine.")
    display(Markdown('\n'.join(lines)))

describe_results()

### 13.1 Discussion

*Read this section together with the generated summary and table above; statements marked **expected** are predictions from theory that the table either confirms or contradicts in your run.*

**Original image.** This is an upper bound: query and reference are pixel-identical, so nearly every keypoint finds its own copy and the homography is close to the identity. Its purpose is to verify that the pipeline is wired correctly, not to measure difficulty.

**Rotation (45°).** *Expected:* recognition succeeds with a large fraction of the original matches, because each descriptor is computed in a frame aligned to the keypoint's dominant orientation. Some loss is expected from interpolation during rotation, the black corners introduced by the enlarged canvas, and keypoints near the image border that are lost.

**Scale (50 %).** *Expected:* recognition succeeds but with fewer keypoints and matches, because halving the resolution removes the finest octave of the scale space; the coarser structures are still detected at a correspondingly smaller scale, which is exactly what scale invariance means.

**Low brightness (× 0.35).** *Expected:* descriptors are largely unaffected because of the unit-length normalisation, but **fewer keypoints are detected** because the DoG contrast threshold is absolute — dimming reduces every DoG response, pushing weak extrema below the threshold. Compare the keypoint column of this condition with the original: this is a detection effect, not a description effect. Histogram equalisation or CLAHE before detection is a standard remedy.

**Viewpoint — synthetic.** A projective warp is exactly representable by a homography, so the geometric model is correct; the difficulty comes from the descriptor, which is only *approximately* invariant to the foreshortening (affine distortion) of the local patch.

**Viewpoint — real held-out photograph.** This is the realistic and hardest case: viewpoint, lighting, weather, season, camera, occluding people and vehicles all change at once, and the landmark is a 3-D structure, so a single homography explains only part of the correspondences. Expect the lowest match and inlier counts here. Failures in this condition are false *rejections* rather than confusions, which is the safer error for a tourism system.

**Unknown landmark.** A small number of ratio-test survivors appear for any image, because some local patterns (window corners, railings, foliage) are generic. They do not agree on a common geometric transformation, so the inlier count stays low and the decision rule rejects the query. This is why the decision cannot rely on the largest match count alone: a ranked list always has a winner, even when the true answer is "none of these".

**Speed.** The time is dominated by brute-force matching against every reference image and grows linearly with the database. For a real deployment with thousands of landmarks this would be replaced by approximate search (FLANN) or vocabulary-tree retrieval followed by geometric verification of only the top candidates.

## 14. Limitations and Error Analysis

1. **Small and automatically collected database.** Five reference photos per landmark cannot cover all viewpoints, seasons and day/night conditions. Automatic search may also include weak references (night shots, partial views), which directly lowers recall. Curating the references or adding more varied ones is the most effective improvement.
2. **Planar model for 3-D structures.** A homography is exact only for planar scenes or pure camera rotation. For large viewpoint changes of 3-D landmarks (the Colosseum's curved façade, the Great Wall winding over hills), correct matches can be classed as outliers. A fundamental-matrix model (`cv2.findFundamentalMat` with RANSAC) is the geometrically correct verification for general 3-D scenes, at the cost of losing the simple polygon localisation.
3. **Intra-class variation — the Great Wall.** "Great Wall of China" is not one object but thousands of kilometres of different sections. A query of a section not represented in the references shares little specific texture with the database, so this class is expected to be the hardest; the categorical identity of such landmarks is better captured by learned global descriptors.
4. **Repetitive structure.** Arches (Colosseum), lattice (Eiffel Tower) and battlements (Great Wall) produce many near-identical local features; the ratio test removes them as ambiguous, which reduces the evidence available and can occasionally create consistent but wrong correspondences.
5. **Thresholds fixed by reasoning, not learned.** $T_m$ and $T_i$ were chosen from the expected chance-match level (Section 8.3). With a larger dataset they should be selected on a validation set (e.g. maximising the true-accept rate at a fixed false-accept rate), and the sensitivity plot shows how the trade-off moves.
6. **Small test set.** With one hold-out image per landmark and two unknown images, the accuracy figures have wide uncertainty; a single test changes accuracy by many percentage points. The results demonstrate behaviour; they are not a statistically precise estimate.
7. **Synthetic conditions are idealised.** Uniform darkening and an exact perspective warp are cleaner than real night photos or real viewpoint changes, which also involve noise, shadows and non-planar effects; synthetic results are therefore optimistic.
8. **Computational cost.** Exact brute-force matching is $O(N_q N_r)$ per reference image; it is fine for tens of images and impractical for thousands without approximate search or indexing.
9. **Localisation polygon semantics.** The polygon outlines the projection of the *entire reference photograph*, not a segmented landmark; with loosely framed references it includes background.

**Possible improvements:** CLAHE before detection for low-light images; RootSIFT (Arandjelović & Zisserman, 2012) for better matching; FLANN or a visual vocabulary for scale; fundamental-matrix verification; per-landmark view clustering of references; learned global descriptors for candidate retrieval followed by SIFT verification.

## 15. Conclusion

A complete SIFT-based landmark recognition system was implemented and evaluated. The reference database stores labelled 128-D SIFT descriptors for five landmarks with multiple photographs each (Tasks 1–2). Queries are encoded with the same pipeline (Task 3), matched with an exact Euclidean brute-force matcher using $k=2$ nearest neighbours and Lowe's ratio test, and landmarks are ranked by their reliable matches (Task 4). The top candidates are verified with a RANSAC homography, which both strengthens the decision and localises the landmark by projecting the reference image's corners; implausible transformations are reported rather than drawn (Task 5). The decision requires both a minimum number of reliable matches and a minimum number of geometrically consistent inliers, so the system can reject landmarks that are not in its database instead of always returning the top-ranked name.

The controlled experiments (Task 6) test the properties that SIFT was designed for — invariance to rotation and scale and robustness to illumination — one factor at a time, and the held-out real photographs and the unknown landmark test the realistic open-set use case. The measured outcomes for this run are summarised in Sections 11–13; the analysis of where the evidence approaches the decision thresholds, together with the limitations in Section 14, shows both the practical value of SIFT matching with geometric verification for a tourism information system and the points where a production system would need a larger reference set, 3-D-aware verification and faster search.

## 16. Viva Preparation: Likely Questions

**Why is SIFT invariant to scale?** Keypoints are detected as extrema of the DoG over a scale space, so each keypoint is assigned the scale $\sigma$ at which its structure responds most strongly; the descriptor window is then sized proportionally to $\sigma$. The same structure seen at half the size is detected at half the $\sigma$ and described over a correspondingly smaller window.

**Why rotation invariant?** The descriptor's gradient orientations are measured relative to the keypoint's dominant orientation (from a 36-bin histogram).

**Why does darkening reduce keypoints if SIFT is illumination invariant?** The *descriptor* is normalised, but *detection* uses an absolute contrast threshold on DoG values, which scale with image intensity.

**Why the ratio test instead of a fixed distance threshold?** Absolute descriptor distances vary greatly between features; the ratio $d_1/d_2$ measures distinctiveness, which separates correct from incorrect matches far better (Lowe, 2004, Fig. 11).

**Why `crossCheck=False`?** Cross-checking returns one mutual best match per feature, but the ratio test needs the two nearest neighbours from `knnMatch(k=2)`; the two filters are alternatives.

**Why match against each reference separately?** With a pooled database, the second neighbour of a correct match often belongs to another photo of the same landmark, and the ratio test would wrongly discard it.

**Why is RANSAC needed, and what is an inlier?** Least squares on all matches is destroyed by outliers. RANSAC fits $\mathbf{H}$ to random 4-point samples and keeps the model with the most correspondences whose reprojection error is below 5 px — those are the inliers.

**Why at least 4 points for a homography?** $\mathbf{H}$ has 9 entries defined up to scale (8 DOF); each correspondence gives 2 equations.

**When is a homography the wrong model?** For a 3-D scene with significant depth seen from two different camera centres. Then the epipolar constraint (fundamental matrix) is the general model.

**Why not decide by the highest match count?** A ranking always has a winner, even for an unknown landmark. Absolute thresholds on matches and inliers provide the "none of these" option.

**What does the polygon represent?** The projection of the reference image's borders into the query under the estimated homography — the query region that corresponds to the reference photo.

**How would you scale this to 10 000 landmarks?** Approximate nearest neighbours (FLANN/HNSW) or a bag-of-visual-words inverted index to shortlist candidates, then RANSAC verification only for the shortlist.

**What is the complexity of brute-force matching?** $O(N_q \times N_r \times 128)$ per reference image, i.e. linear in database size.

## 17. Requirement-Compliance Checklist

| # | Requirement (assignment) | Where it is met |
|---|---|---|
| T1 | Database of ≥ 5 famous landmarks | §5 — Eiffel Tower, Taj Mahal, Colosseum, Statue of Liberty, Great Wall; assertion in §5.1 |
| T1 | Multiple images per landmark, different angles/distances | §5 — 5 references per landmark (+1 held-out); contact sheet |
| T1 | Separate folder per landmark | `landmark_db/<Landmark_Name>/` |
| T2.1 | Load each reference image, convert to grayscale | `load_image`, `extract_sift` (§6) |
| T2.2 | Detect SIFT keypoints | `cv2.SIFT_create().detectAndCompute` |
| T2.3 | Extract descriptors | 128-D float32 descriptors, shape asserted |
| T2.4 | Display references with keypoints | §6.1 grid figure (rich keypoints) |
| T2.5 | Store descriptors with labels | `reference_db` + `outputs/sift_descriptor_database.npz` (verified reload) |
| T3.1–3.4 | Load query, grayscale, detect, describe | `process_query` (§7) |
| T3.5 | Display query with keypoints | `show_query` (§7), panel (2) of every result |
| T4.1 | Brute-Force matcher, Euclidean | `cv2.BFMatcher(cv2.NORM_L2)` |
| T4.2 | k-NN with k = 2 | `knnMatch(..., k=2)` |
| T4.3 | Lowe's ratio test | `ratio_test_matches`, τ = 0.75 |
| T4.4 | Count reliable matches per landmark | `rank_landmarks` (best reference per landmark) |
| T4.5 | Rank candidates | sorted ranking, table + bar chart |
| T4.6 | Display most likely landmark + matches | result figure header and panel (4–5) |
| T4 | Threshold to reject unknown; not max-count only | matches ≥ 20 **and** inliers ≥ 15 (§8.3, `recognize`) |
| T5.1 | Corresponding keypoints ref ↔ query | `localize` builds src/dst point arrays |
| T5.2 | Homography with RANSAC when enough matches | `cv2.findHomography(..., cv2.RANSAC, 5.0)`, ≥ 10 matches |
| T5.3 | Project four reference corners | `cv2.perspectiveTransform` |
| T5.4 | Draw polygon | `cv2.polylines`, panel (6) |
| T5.5 | Display reliable matches and inliers | result header, tables |
| T5 | Message instead of invalid polygon | plausibility checks + message in panel (6) |
| T6 | Original / rotated / resized / low-brightness / viewpoint / unknown | §11 — every condition for every landmark (+ synthetic viewpoint) |
| T6 | Record prediction, keypoints, matches, decision, time | `experimental_results.csv`, table §11.2 |
| Output 1–7 | Query, keypoints, matches, name, matches count, localisation, accept/reject message | `show_result` panels (1)–(7) |
| Deliverable | Source code | this notebook (File → Download → .py also available) |
| Deliverable | Organised image database | `landmark_db/`, `queries/` in the submission zip |
| Deliverable | Screenshots of output | `outputs/screenshots/*.png` (one per test + analysis figures) |
| Deliverable | Table of experimental results | §11.2, `outputs/experimental_results.csv` |
| Deliverable | Discussion of performance and limitations | §13, §14 |

## 18. Packaging the Deliverables
Creates `landmark_lab_submission.zip` with the image database, the queries, the descriptor database, the results table, the image source manifest and all screenshots, and downloads it in Colab. Download the notebook itself from **File → Download → Download .ipynb** (and **.py** for plain source code).

In [ ]:
zip_path = DATA_ROOT.parent / 'landmark_lab_submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(DATA_ROOT.rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to(DATA_ROOT.parent))
size_mb = zip_path.stat().st_size / 1e6
print(f'{zip_path} — {size_mb:.1f} MB')
for sub in ['landmark_db', 'queries', 'outputs', 'outputs/screenshots']:
    n = len([p for p in (DATA_ROOT / sub).rglob('*') if p.is_file()])
    print(f'  {sub:22s} {n:4d} files')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

## 19. References

1. D. G. Lowe, "Distinctive Image Features from Scale-Invariant Keypoints," *International Journal of Computer Vision*, 60(2), pp. 91–110, 2004.
2. D. G. Lowe, "Object Recognition from Local Scale-Invariant Features," *Proc. IEEE International Conference on Computer Vision (ICCV)*, pp. 1150–1157, 1999.
3. M. A. Fischler and R. C. Bolles, "Random Sample Consensus: A Paradigm for Model Fitting with Applications to Image Analysis and Automated Cartography," *Communications of the ACM*, 24(6), pp. 381–395, 1981.
4. R. Hartley and A. Zisserman, *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004 (homography estimation, DLT, RANSAC).
5. R. Arandjelović and A. Zisserman, "Three things everyone should know to improve object retrieval," *Proc. IEEE CVPR*, 2012 (RootSIFT).
6. OpenCV documentation — *Introduction to SIFT*; *Feature Matching*; *Feature Matching + Homography to find Objects*. https://docs.opencv.org/4.x/
7. Wikimedia Commons — source of the reference and query photographs; individual licences and authors are listed in `outputs/image_manifest.csv`. https://commons.wikimedia.org